# Merge per-paper extractions into one annotation-tool file (append-only)

Combines the per-paper extraction files (e.g. `PMC*.json`) in one annotator's
folder into a single `<task>_<annotator>.json` that the annotation app loads as
one continuous document. Each sentence is tagged with a `doc` provenance field
(the source paper's file stem), shown in the app header and at each boundary.

## Workflow
1. Drop new `PMC*.json` files into `extracted_triplets/<annotator>/`.
2. Re-run this notebook.
3. The annotator keeps working — on their next click the app picks up the new
   sentences automatically (no restart), and their annotations keep appending to
   `<task>_<annotator>_annotated.json`. You never edit that output file.

## Append-only guarantee
Papers **already** in the merged file are kept **verbatim, in their existing
order** — their sentence indices never move. Only papers not yet present are
**appended at the end**. This is what keeps an in-progress annotator's saved
decisions (keyed by sentence index) correctly aligned. Re-running with no new
files is a no-op.

> **Note:** because existing papers are frozen, re-extracting a paper that is
> *already merged* will **not** be picked up (that would shift indices and
> invalidate annotations). Adding brand-new papers is always safe.

> Run from the `annotation_app/` folder (or adjust `DATA_DIR`).

In [ ]:
# ---- Configuration ----
ANNOTATOR  = "mark"                # annotator name = subfolder under DATA_DIR
TASK       = "triplets"            # "relations" or "triplets" -> <TASK>_<ANNOTATOR>.json
DATA_DIR   = "extracted_triplets"  # folder holding <annotator>/ subfolders
PAPER_GLOB = "PMC*.json"           # which files to merge (the per-paper extractions)
NEW_ORDER  = "mtime"               # order to append NEW papers in: "mtime" or "name"

In [2]:
import json, os, tempfile
from pathlib import Path

folder = Path(DATA_DIR) / ANNOTATOR
assert folder.is_dir(), f"No such folder: {folder.resolve()}"

out_name = f"{TASK}_{ANNOTATOR}.json"
out_path = folder / out_name

# 1. Papers ALREADY merged (frozen, in their current order).
existing = json.load(open(out_path, encoding="utf-8")) if out_path.exists() else []
already = []
for it in existing:
    d = it.get("doc", "")
    if d and d not in already:
        already.append(d)

# 2. Per-paper files present in the folder (excluding the merged output itself).
present = [p for p in folder.glob(PAPER_GLOB) if p.name != out_name]

# 3. NEW papers = present but not yet merged.
new_files = [p for p in present if p.stem not in set(already)]
if NEW_ORDER == "mtime":
    new_files.sort(key=lambda p: p.stat().st_mtime)
elif NEW_ORDER == "name":
    new_files.sort(key=lambda p: p.name)
else:
    raise ValueError(f"Unknown NEW_ORDER: {NEW_ORDER!r}")

print(f"Already merged ({len(already)} paper(s), frozen):", already or "—")
print(f"New to append   ({len(new_files)} paper(s)):", [p.stem for p in new_files] or "—")
# Heads-up: a merged paper whose source file is gone (kept anyway, indices preserved).
gone = [d for d in already if not (folder / f"{d}.json").exists()]
if gone:
    print("Note: merged papers with no source file (kept as-is):", gone)

Already merged (0 paper(s), frozen): —
New to append   (12 paper(s)): ['PMC11004549', 'PMC12571284', 'PMC4220923', 'PMC5192166', 'PMC5358856', 'PMC6023484', 'PMC6650471', 'PMC6920850', 'PMC7042760', 'PMC7940314', 'PMC8216903', 'PMC9484017']


In [3]:
REQUIRED = {"text", "spans", "triplets"}

merged = list(existing)          # frozen prefix: existing indices never move
for p in new_files:
    doc = p.stem                 # provenance tag, e.g. "PMC11004549"
    items = json.load(open(p, encoding="utf-8"))
    assert isinstance(items, list), f"{p.name} is not a JSON list"
    for it in items:
        if not REQUIRED <= set(it):
            raise ValueError(f"{p.name}: an item is missing {REQUIRED - set(it)}")
        it = dict(it)
        it["doc"] = doc
        merged.append(it)
    n_trip = sum(len(it.get("triplets", [])) for it in items)
    print(f"  appended {p.name:24} {len(items):4d} sentences  {n_trip:5d} triplets")

if not new_files:
    print("Nothing new to append — merged file left unchanged.")

  appended PMC11004549.json           88 sentences    270 triplets
  appended PMC12571284.json          154 sentences    526 triplets
  appended PMC4220923.json           130 sentences    197 triplets
  appended PMC5192166.json            86 sentences    172 triplets
  appended PMC5358856.json           110 sentences    178 triplets
  appended PMC6023484.json            57 sentences    230 triplets
  appended PMC6650471.json           146 sentences    436 triplets
  appended PMC6920850.json           222 sentences    256 triplets
  appended PMC7042760.json           116 sentences    179 triplets
  appended PMC7940314.json           119 sentences    273 triplets
  appended PMC8216903.json           126 sentences    276 triplets
  appended PMC9484017.json           139 sentences    406 triplets


In [4]:
# Atomic write (temp file + os.replace) so the app never reads a half-written
# file if it happens to reload while this runs.
def atomic_dump(obj, path: Path):
    fd, tmp = tempfile.mkstemp(prefix=".tmp_", dir=str(path.parent), suffix=".json")
    try:
        with os.fdopen(fd, "w", encoding="utf-8") as fh:
            json.dump(obj, fh, ensure_ascii=False, indent=2)
        os.replace(tmp, path)
    finally:
        if os.path.exists(tmp):
            os.remove(tmp)

atomic_dump(merged, out_path)

n_trip = sum(len(it["triplets"]) for it in merged)
n_docs = len({it.get("doc", "") for it in merged})
print(f"Wrote {out_path}")
print(f"  {len(merged)} sentences, {n_trip} triplets, {n_docs} papers")
print("\nPaper boundaries (sentence # where each paper starts):")
prev = None
for i, it in enumerate(merged):
    if it.get("doc") != prev:
        print(f"  #{i + 1:<5} {it.get('doc')}")
        prev = it.get("doc")

Wrote extracted_triplets/mark/triplets_mark.json
  1493 sentences, 3399 triplets, 12 papers

Paper boundaries (sentence # where each paper starts):
  #1     PMC11004549
  #89    PMC12571284
  #243   PMC4220923
  #373   PMC5192166
  #459   PMC5358856
  #569   PMC6023484
  #626   PMC6650471
  #772   PMC6920850
  #994   PMC7042760
  #1110  PMC7940314
  #1229  PMC8216903
  #1355  PMC9484017
